# PoetryDuel-GPT v4 — Thất Ngôn + Rhyme Constraint

**749K window-1 pairs (Lục Bát + Thất Ngôn), example-aligned batches, rhyme beam constraint, overgeneration fix.**

| Step | Time |
|---|---|
| Download data + train tokenizer | ~2 min |
| Training (10K steps) | ~2.5 hr T4 |
| Generate poetry | ~1 min |

### v4 Changes from v3:
- **T1: Thất Ngôn (7→7)** — 41K bảy chữ poems now included (~208K training pairs)
- **P1: Rhyme Constraint** — beam-time rhyme forcing at pos 6/7 of 2nd line
- **P2: Overgeneration Fix** — auto-truncates >2 line outputs to first valid couplet

In [ ]:
# @title 1. Clone Repo + Install
!git clone https://github.com/weseegod/poetry-dual-gpt.git /content/poetry-dual-gpt 2>/dev/null
%cd /content/poetry-dual-gpt
!git pull origin main

!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pandas tokenizers tqdm datasets
!mkdir -p checkpoints

import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print('\n\u26a0\ufe0f  Run cells in order.')


In [ ]:
# @title 2. Download Data + Train Tokenizer (~2 min)
from google.colab import drive
drive.mount('/content/drive')

# Copy data.zip from Drive
!cp /content/drive/MyDrive/poetry-dual-gpt/data.zip /content/poetry-dual-gpt/ 2>/dev/null
!unzip -o data.zip

# If zip preserved dirs: corpus is at data/doi_tho_corpus.txt
# If old zip (flat): corpus is at doi_tho_corpus.txt — move it
!mkdir -p data
!mv doi_tho_corpus.txt data/doi_tho_corpus.txt 2>/dev/null
!ls data/

# Train BPE tokenizer on v4 corpus (Lục Bát + Thất Ngôn)
%cd /content/poetry-dual-gpt
!python src/train_bpe.py --corpus data/doi_tho_corpus.txt

# Verify
from tokenizers import Tokenizer
tok = Tokenizer.from_file('tokenizer/poetry_bpe.model')
print(f'Vocab: {tok.get_vocab_size():,}')
for name in ['<|pad|>', '<|start|>', '<|reply|>', '<|end|>', '[DOI_THO]', '<|linebreak|>']:
    n = len(tok.encode(name).ids)
    print(f'  {name:20s} subwords={n} {"OK" if n==1 else "FAIL"}')

# v4: Verify both genres present
import re
with open('data/doi_tho_corpus.txt', encoding='utf-8') as f:
    corpus = f.read()
lb_count = corpus.count('[TONE:BBBBBB]')
tn_count = corpus.count('[TONE:BBBBBBB]')
print(f'\nLục Bát pairs (6-char tone): ~{lb_count:,}')
print(f'Thất Ngôn pairs (7-char tone): ~{tn_count:,}')

print("\nReady for training.")


In [ ]:
# @title 3. Train — Example-Aligned, v4 Corpus (~2.5 hr T4)
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> T4 GPU"

%cd /content/poetry-dual-gpt
import glob

# v4: 749K window-1 doi tho pairs (Lục Bát + Thất Ngôn), example-aligned
CORPUS = 'data/doi_tho_corpus.txt'

latest = sorted(glob.glob("checkpoints/doi_tho_step_*.pt"))
if latest:
    print(f"Resuming from {latest[-1]}")
    !python src/train.py --mode train --name doi_tho_ --corpus {CORPUS} --resume {latest[-1]}
else:
    !python src/train.py --mode train --name doi_tho_ --corpus {CORPUS}

print("\nTraining complete -> checkpoints/doi_tho_best.pt")


In [ ]:
# @title Verify: Test Generation (Lục Bát + Thất Ngôn)
%cd /content/poetry-dual-gpt
import os

ckpt = 'checkpoints/doi_tho_best.pt'
if not os.path.exists(ckpt):
    import glob
    candidates = sorted(glob.glob('checkpoints/doi_tho_*.pt'), reverse=True)
    for c in candidates:
        if os.path.getsize(c) > 1000000:
            ckpt = c
            break

if not os.path.exists(ckpt):
    print('No checkpoint found. Run training first.')
else:
    print(f'Testing: {ckpt}')
    
    print('\n--- Lục Bát Đối Thơ ---')
    !python src/sample.py --checkpoint {ckpt} \
        --prompt "Than em nhu chen lua dong | Phat pho duoi ngon nang hong ban mai" \
        --num_samples 2 --device cuda
    
    print('\n--- Thất Ngôn Đối Thơ ---')
    !python src/sample.py --checkpoint {ckpt} \
        --prompt "Ao thu lanh leo nuoc trong veo | Mot chiec thuyen cau be teo teo" \
        --num_samples 2 --device cuda
    
    print('\nExpected: Lục Bát (6+8), Thất Ngôn (7+7) with rhyme constraint.')


In [ ]:
# @title Generate Poetry (custom prompt)
%cd /content/poetry-dual-gpt

# Lục Bát (use | between lines)
!python src/sample.py \
    --checkpoint checkpoints/doi_tho_best.pt \
    --prompt "Than em nhu chen lua dong | Phat pho duoi ngon nang hong ban mai" \
    --temperature 0.75 \
    --top_p 0.92 \
    --num_samples 2 \
    --device cuda

# Thất Ngôn
!python src/sample.py \
    --checkpoint checkpoints/doi_tho_best.pt \
    --prompt "Ao thu lanh leo nuoc trong veo | Mot chiec thuyen cau be teo teo" \
    --temperature 0.75 \
    --top_p 0.92 \
    --num_samples 2 \
    --device cuda


In [ ]:
# @title Save to Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/poetry-dual-gpt'
!mkdir -p {DRIVE_DIR}/checkpoints

!cp -v tokenizer/poetry_bpe.model {DRIVE_DIR}/
import glob
for ckpt in sorted(glob.glob('checkpoints/doi_tho_*.pt')):
    !cp -v {ckpt} {DRIVE_DIR}/checkpoints/
print('All saved to Drive -> poetry-dual-gpt/')
